# Lab 2 · Advanced RAG (free Kaggle T4)

**Layer 3 · Data — Chapter 2 (the advanced pipeline).** Lab 1 built naive RAG. Here
we add the four levers that make retrieval *good* — then **measure** that they help:

1. **Metadata filtering** — restrict the search by tag (topic / tenant / date)
2. **Hybrid search** — fuse vector (meaning) with BM25 (exact terms)
3. **Query understanding** — rewrite + multi-query so phrasing doesn't matter
4. **Reranking** — a cross-encoder re-scores query+chunk together

Same **k8s Q&A dataset** and open models as Lab 1.

**Settings:** Accelerator = **GPU T4 x2** (not P100), Internet = **On**, then **Run
All**. Embedder + BM25 + reranker run on CPU; only the LLM uses the GPU.

*Data: [`ItshMoh/kubernetes_qa_pairs`](https://huggingface.co/datasets/ItshMoh/kubernetes_qa_pairs).*

In [ ]:
import warnings, logging, os
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
for _n in ('huggingface_hub', 'sentence_transformers', 'transformers', 'datasets'):
    logging.getLogger(_n).setLevel(logging.ERROR)

!pip install -q faiss-cpu sentence-transformers datasets rank-bm25 2>/dev/null

import faiss, textwrap
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder

## 0 · Rebuild the index (recap of Lab 1)

Pull the dataset, embed every answer with **`bge-small`** (CPU), load a flat **FAISS** index. Self-contained — no dependency on Lab 1.

In [ ]:
from datasets import load_dataset
ds = load_dataset('ItshMoh/kubernetes_qa_pairs', split='train')
df = ds.to_pandas()

EMB_ID = 'BAAI/bge-small-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
embedder = SentenceTransformer(EMB_ID, device='cpu')

chunks = [{'id': f'qa-{i}', 'text': r['answer'], 'question': r['question'], 'topic': r['topic']}
          for i, r in enumerate(ds) if (r['answer'] or '').strip()]
vecs = embedder.encode([c['text'] for c in chunks], normalize_embeddings=True,
                       convert_to_numpy=True, batch_size=128).astype('float32')
index = faiss.IndexFlatIP(vecs.shape[1]); index.add(vecs)

def retrieve(query, k=3):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = index.search(qv, k)
    return [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0])]

print('indexed', index.ntotal, 'answer-chunks |', df['topic'].nunique(), 'topics')

## 1 · Metadata filtering

Every chunk carries metadata (here: `topic`). Filtering at retrieval is what enforces *permissions* and *freshness* in production (only this tenant, only since 2026). Flat FAISS has no native filter, so we over-fetch and filter in Python (a real vector DB does this inside the search).

In [ ]:
from collections import Counter
print('chunks carry metadata, e.g.:', {k: chunks[0][k] for k in ('id','topic')})
print('top topics:', Counter(c['topic'] for c in chunks).most_common(5))

# Flat FAISS has no native metadata filter: over-fetch, then filter in Python.
def retrieve_filtered(query, k=3, topic=None):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = index.search(qv, 50)
    hits = [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0])]
    if topic: hits = [h for h in hits if h['topic'] == topic]
    return hits[:k]

TOPIC = Counter(c['topic'] for c in chunks).most_common(1)[0][0]
q = 'how do I configure access and permissions?'
print(f'\nno filter   :', [h['topic'] for h in retrieve(q, 5)])
print(f'topic={TOPIC!r}:', [h['topic'] for h in retrieve_filtered(q, 5, topic=TOPIC)])

## 2 · Hybrid search (vector ∪ BM25)

Dense vectors capture *meaning* but can under-rank **exact terms** (component names, error codes). **BM25** is keyword search. We fuse the two rankings with **Reciprocal Rank Fusion** — meaning *and* literal matches.

In [ ]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([c['text'].lower().split() for c in chunks])

def hybrid(query, k=5, depth=30):
    # dense ranking
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    _, di = index.search(qv, depth); dense = [int(i) for i in di[0]]
    # sparse (BM25) ranking
    bm = list(np.argsort(bm25.get_scores(query.lower().split()))[::-1][:depth])
    # Reciprocal Rank Fusion: fuse the two rankings
    rrf = {}
    for rank, i in enumerate(dense): rrf[i] = rrf.get(i, 0) + 1 / (60 + rank)
    for rank, i in enumerate(bm):    rrf[i] = rrf.get(i, 0) + 1 / (60 + rank)
    top = sorted(rrf, key=rrf.get, reverse=True)[:k]
    return [{**chunks[i], 'rrf': rrf[i]} for i in top]

# A query with an exact term dense search can under-rank, but BM25 nails:
q = 'what is a kubelet?'
print('dense-only top-3:', [h['topic'] for h in retrieve(q, 3)])
print('hybrid    top-3:', [h['topic'] for h in hybrid(q, 3)])

## 3 · Query understanding (multi-query)

The user's phrasing may not match the docs. Have the LLM produce alternate phrasings, retrieve for each, and **union** the results. (This cell loads **Qwen2.5-3B** — ~6 GB on the T4.)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto').eval()

def generate(messages, max_new_tokens=200):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print('LLM on', model.device)

In [ ]:
# The user's phrasing may not match the docs. Ask the LLM for alternates,
# retrieve for each, and UNION the results (multi-query).
def rewrites(query, n=3):
    msg = [{'role': 'user', 'content':
            f'Give {n} alternative search phrasings of this Kubernetes question, '
            f'one per line, no numbering:\n{query}'}]
    lines = [l.strip('-* ').strip() for l in generate(msg, 120).split(chr(10)) if l.strip()]
    return [query] + lines[:n]

def multi_query(query, k=5):
    seen, pool = set(), []
    for q in rewrites(query):
        for h in hybrid(q, k):
            if h['id'] not in seen:
                seen.add(h['id']); pool.append(h)
    return pool

q = 'pod stuck pending'
print('rewrites:'); [print('  -', r) for r in rewrites(q)]
print(f'\nsingle hybrid -> {len(hybrid(q, 5))} chunks | multi-query -> {len(multi_query(q, 5))} chunks')

## 4 · Reranking

Vector/BM25 are fast but blunt. A **cross-encoder** reads query+chunk *together* and re-scores. Over-fetch cheaply, then keep the best few — the biggest lever after chunking.

In [ ]:
reranker = CrossEncoder('BAAI/bge-reranker-base', device='cpu')

# Over-fetch with hybrid, then the cross-encoder re-reads query+chunk together.
q = 'How does Kubernetes clean up unused container images?'
cands = hybrid(q, k=10)
print('BEFORE rerank (RRF order), top 3:')
for h in cands[:3]:
    print(f"  {h['topic']}: {h['text'][:70]}...")
scores = reranker.predict([[q, h['text']] for h in cands])
for h, s in zip(cands, scores): h['rr'] = float(s)
for h in sorted(cands, key=lambda h: h['rr'], reverse=True)[:3]:
    print('AFTER:', f"rr={h['rr']:.2f}  {h['topic']}: {h['text'][:60]}...")

## 5 · The full advanced answer

Hybrid retrieve → rerank → keep top chunks → grounded, cited **Qwen** answer (and a refusal when the context can't answer).

In [ ]:
SYS = ('You are a Kubernetes assistant. Use ONLY the CONTEXT to answer, and cite the chunk ids in [brackets]. '
       "Do not use any outside knowledge. If the answer is not in the CONTEXT, reply exactly: I don't know.")

def answer(query, keep=3):
    cands = hybrid(query, k=20)                                  # hybrid candidates
    scores = reranker.predict([[query, h['text']] for h in cands])
    for h, s in zip(cands, scores): h['rr'] = float(s)
    hits = sorted(cands, key=lambda h: h['rr'], reverse=True)[:keep]   # rerank -> best few
    ctx = '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
    ans = generate([{'role': 'system', 'content': SYS},
                    {'role': 'user', 'content': f'CONTEXT:\n{ctx}\n\nQUESTION: {query}'}])
    return ans, hits

ans, hits = answer('What is Role-Based Access Control (RBAC) in Kubernetes?')
print('loaded chunks:', [h['id'] for h in hits])
print('\nANSWER:\n', textwrap.fill(ans, 90))
print('\nREFUSAL:', answer('What is the capital of France?')[0])

## 6 · Does it actually help? — evaluate retrieval

Each question's own answer is its gold chunk, so we can measure **recall@3** (is the gold chunk in the top 3?) and **MRR** (1/rank of the gold chunk). We compare three retrievers: vector-only (Lab 1) → + hybrid → + rerank. Watch reranking lift **MRR** most — it puts the right chunk *first*.

In [ ]:
EVAL_N = 40          # raise for a tighter estimate (slower: reranks EVAL_N x depth pairs on CPU)
sample = chunks[:EVAL_N]

def rank_of(gold_id, ordered):
    for r, h in enumerate(ordered, 1):
        if h['id'] == gold_id: return r
    return None

def score(retriever):
    rr_sum = hit3 = 0
    for c in sample:
        r = rank_of(c['id'], retriever(c['question']))
        if r: rr_sum += 1 / r; hit3 += (r <= 3)
    return hit3 / len(sample), rr_sum / len(sample)

def vec(q):    return retrieve(q, 10)
def hyb(q):    return hybrid(q, 10)
def hyb_rr(q):
    cands = hybrid(q, 10)
    s = reranker.predict([[q, h['text']] for h in cands])
    for h, v in zip(cands, s): h['rr'] = float(v)
    return sorted(cands, key=lambda h: h['rr'], reverse=True)

print(f'over {EVAL_N} questions    recall@3    MRR')
for name, fn in [('vector-only', vec), ('+ hybrid (BM25)', hyb), ('+ hybrid + rerank', hyb_rr)]:
    r, m = score(fn)
    print(f'  {name:<18} {r:5.2f}     {m:5.2f}')

## Takeaways

- **Four levers, each fixing a Lab-1 gap:** metadata filtering (permissions/freshness),
  hybrid (exact terms), query understanding (phrasing), reranking (ordering).
- **Reranking lifts MRR more than recall** — it doesn't find *more* right chunks,
  it puts the right one *first*, which is what the model actually reads.
- **Always measure.** recall@k / MRR tell you *where* a failure is; "the demo
  answered well" does not. Evaluate retrieval separately from generation.
- Next: agentic RAG — the agent *decides* whether/what/when to retrieve (Layer 4).